# SourceSpec Demo

SourceSpec2 has been redesigned to allow interactive running.

To do so, we need to:
- configure sourcespec
- collect 4 pieces of information:
    - event information (as instance of :class:`sourcespec.ssp_event.SSPEvent`)
    - phase picks (list of :class:`sourcespec.ssp_event.Pick`)
    - trace data (instance of :class:`obspy.Trace`)
    - instrument response (instance of :class:`obspy.Inventory`)

In this demo, we run one of the examples in the `sourcespec_testruns` package

In [ ]:
import os
import sourcespec2

In [ ]:
#testrun_folder = r'C:\Users\kris\Documents\Python\cloned_repos\sourcespec_testruns\test_CDSA'
testrun_folder = os.path.realpath('../test_CDSA')

## Configuration

Load the global configuration object

In [ ]:
from sourcespec2.setup import config

There is a dedicated notebook explaining sourcespec configuration in more detail.

Here, we will just read an existing configuration file:

In [ ]:
cfg_file = os.path.join(testrun_folder, 'source_spec2.conf')
config.read(cfg_file)
config.running_from_command_line = False

Inspect configuration parameters

In [ ]:
config

## Collect required data

The required information may either be provided as appropriate objects or we can use sourcespec's builtin functionality. Here, we demonstrate the 2nd option.

Set paths to different pieces of information:
- event information and phase picks: Quakeml file
- trace data: miniseed file
- instrument response: StationXML file

This information is set in config.options, which represents sourcespec's command-line arguments

In [ ]:
config.workdir = testrun_folder
data_folder = os.path.join(testrun_folder, 'data')
config.options.qml_file = os.path.join(data_folder, 'cdsa20100421051050GL.xml')
config.options.trace_path = [os.path.join(data_folder, 'cdsa20100421051050GL.mseed')]
config.options.station_metadata = os.path.join(data_folder, 'inventory.xml')
# FIXME: station_metadata is also present in config, which takes precedence
config.station_metadata = config.options.station_metadata

Inspect options

In [ ]:
config.options

Read event and phase picks

In [ ]:
from sourcespec2.input import read_event_and_picks
event, picks = read_event_and_picks()
print(event)
print(len(picks))

Read instrument response

In [ ]:
from sourcespec2.input import read_station_metadata
inventory = read_station_metadata()
print(inventory)

Read traces

In [ ]:
from sourcespec2.input.traces import read_traces
st = read_traces()
print(st)

## Run sourcespec

In [ ]:
from sourcespec2.source_spec import (ssp_run, ssp_output, ssp_clear_state)

Setup logging

In [ ]:
import sys, logging

logging.basicConfig(
    #format='%(asctime)s [%(levelname)s] %(name)s - %(message)s',
    level=logging.INFO,
    #datefmt='%Y-%m-%d %H:%M:%S',
    stream=sys.stdout,
)

Run sourcespec

In [ ]:
config.options.outdir = None
result = ssp_run(st, inventory, event, picks)

Unpack and print results

In [ ]:
(proc_st, spec_st, specnoise_st, weight_st, sspec_output) = result

In [ ]:
sspec_output.mean_values()

In [ ]:
sspec_output.mean_nobs()

In [ ]:
sspec_output.reference_summary_parameters()

In [ ]:
# FIXME: maybe implement this as pretty_print method in SourceSpecOutput?
from sourcespec2.ssp_output import _dict2yaml
lines = _dict2yaml(sspec_output)
print(lines)

## Output

Generate all output according to configuration

In [ ]:
#config.options.outdir = r'C:\Temp\sourcespec'
config.options.outdir = os.path.join(testrun_folder, 'output')
ssp_output(st, proc_st, spec_st, specnoise_st, weight_st, sspec_output)

Show individual plots interactively, without saving

In [ ]:
config.plot_show = True
config.plot_save = False

In [ ]:
from sourcespec2.ssp_plot_spectra import plot_spectra
plot_spectra(spec_st, specnoise_st, plot_type='regular')

In [ ]:
from sourcespec2.ssp_plot_traces import plot_traces
plot_traces(st, suffix='raw')

In [ ]:
from sourcespec2.ssp_plot_stacked_spectra import plot_stacked_spectra
plot_stacked_spectra(spec_st, weight_st, sspec_output)

In [ ]:
from sourcespec2.ssp_plot_params_stats import box_plots
box_plots(sspec_output)

In [ ]:
from sourcespec2.ssp_plot_stations import plot_stations
result = plot_stations(sspec_output)

It is best to clear state before a next run

In [ ]:
ssp_clear_state(reset_config=False)